# Folium

In [ ]:
import folium
import numpy as np
import pandas as pd

In [ ]:
# Crear un mapa de Bogotá
m_bogota = folium.Map(
 location = [4.7110, -74.0721],
 zoom_start = 12,
 tiles='CartoDB positron'
)

In [ ]:
# Mostrar mapa
m_bogota

In [ ]:
# Puntos de interés en Bogotá
lugares = {
    'Plaza de Bolívar':       [4.5981, -74.0760],
    'Aeropuerto El Dorado':   [4.7016, -74.1469],
    'Parque Simón Bolívar':   [4.6583, -74.0939],
    'Universidad Nacional':   [4.6358, -74.0828],
    'Zona Rosa (Chapinero)':  [4.6657, -74.0530],
    'Universidad Javeriana': [4.627620457872371, -74.06474737471112]
}

# Iconos diferenciados
iconos = {
    'Plaza de Bolívar':       ('red',    'university', 'fa'),
    'Aeropuerto El Dorado':   ('blue',   'plane',      'fa'),
    'Parque Simón Bolívar':   ('green',  'tree',       'fa'),
    'Universidad Nacional':   ('purple', 'book',       'fa'),
    'Zona Rosa (Chapinero)':  ('orange', 'coffee',     'fa'),
    'Universidad Javeriana':   ('purple', 'book',       'fa'),
}

for nombre, coordenadas in lugares.items():
    icono_color, icono_tipo, icono_icono = iconos[nombre]
    folium.Marker(
        location = coordenadas,
        popup = folium.Popup(f'<b>{nombre}</b><br>Lat: {coordenadas[0]}<br>Lon: {coordenadas[1]}',
                           max_width=200),
        icon= folium.Icon(color=icono_color, icon=icono_icono, prefix=icono_tipo)).add_to(m_bogota)
m_bogota



In [ ]:
m_bogota.save('bogota.html')

In [ ]:
train=pd.read_excel('Datos Taller individual - 2410 (Estudiantes).xlsx')

In [ ]:
n_tiendas = 50
# Coordenadas aleatorias alrededor del centro de Bogotá
lats = np.random.uniform(4.58, 4.75, n_tiendas)
lons = np.random.uniform(-74.15, -74.00, n_tiendas)

# Submuestra de los datos de entrenamiento
muestra = train.sample(n_tiendas, random_state=42).reset_index(drop=True)

# Escala de colores manual: rojo (bajo) → verde (alto)
def valor_a_color(valor, vmin, vmax):
    ratio = (valor - vmin) / (vmax - vmin)
    r = int(255 * (1 - ratio))
    g = int(200 * ratio)
    return f'#{r:02X}{g:02X}50'

v_min = muestra['ropamujer'].min()
v_max = muestra['ropamujer'].max()

m_circles = folium.Map(location=[4.660, -74.075], zoom_start=12,
                       tiles='CartoDB dark_matter')

for i in range(n_tiendas):
    radio = 5 + (muestra.loc[i, 'nomina'] - train['nomina'].min()) / \
                 (train['nomina'].max() - train['nomina'].min()) * 15

    color_hex = valor_a_color(muestra.loc[i, 'ropamujer'], v_min, v_max)

    popup_html = (
        f"<b>Local #{muestra.loc[i,'idloc']}</b><br>"
        f"Tamaño: {muestra.loc[i,'tamamer']}<br>"
        f"Nómina: ${muestra.loc[i,'nomina']:,}<br>"
        f"Ropa mujer: ${muestra.loc[i,'ropamujer']:,.0f}"
    )

    folium.CircleMarker(
        location=[lats[i], lons[i]],
        radius=radio,
        color='white',
        weight=0.5,
        fill=True,
        fill_color=color_hex,
        fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"Local {muestra.loc[i,'idloc']}"
    ).add_to(m_circles)

m_circles